# English EPUB Extraction & Alignment

Extract the English translation of *The Book of Disquiet* and verify alignment 
with the Portuguese fragments.

In [2]:
import ebooklib
from ebooklib import epub
from bs4 import BeautifulSoup
import json
import re

book = epub.read_epub('../data/book.epub')

# Build href to content mapping
href_to_content = {}
for item in book.get_items_of_type(ebooklib.ITEM_DOCUMENT):
    content = item.get_content()
    if isinstance(content, bytes):
        content = content.decode('utf-8', errors='ignore')
    href_to_content[item.get_name()] = content

# Flatten TOC
all_links = []
for item in book.toc:
    if isinstance(item, tuple):
        section, links = item
        all_links.extend(links)
    elif hasattr(item, 'href'):
        all_links.append(item)

# Extract numbered fragments (English uses plain digits: "1", "2", "3")
english_sections = {}
seen_hrefs = set()

for link in all_links:
    href = link.href.split('#')[0]
    if href in seen_hrefs or href not in href_to_content:
        continue
    seen_hrefs.add(href)
    
    title = link.title if hasattr(link, 'title') else ''
    if not title.isdigit():
        continue
    
    soup = BeautifulSoup(href_to_content[href], 'html.parser')
    paragraphs = []
    for p_tag in soup.find_all('p'):
        text = p_tag.get_text().strip()
        text = re.sub(r'\s+', ' ', text)
        if text and len(text) > 20:
            paragraphs.append(text)
    
    if paragraphs:
        english_sections[title] = paragraphs

print(f"✅ Extracted {len(english_sections)} English fragments")

✅ Extracted 481 English fragments


In [3]:
with open('../data/english_paragraphs.json', 'w', encoding='utf-8') as f:
    json.dump(english_sections, f, ensure_ascii=False, indent=2)
print("Saved to data/english_paragraphs.json")

Saved to data/english_paragraphs.json


## Verify Alignment

Check that Portuguese and English fragments match by number.

In [4]:
with open('../data/fragments.json', 'r', encoding='utf-8') as f:
    pt = json.load(f)

print(f"Portuguese fragments: {len(pt)}")
print(f"English fragments: {len(english_sections)}")

pt_keys = set(pt.keys())
en_keys = set(english_sections.keys())

missing_en = sorted(pt_keys - en_keys, key=int)
missing_pt = sorted(en_keys - pt_keys, key=int)

if missing_en:
    print(f"\n⚠️ Missing in English: {missing_en[:10]}")
if missing_pt:
    print(f"\n⚠️ Missing in Portuguese: {missing_pt[:10]}")
if not missing_en and not missing_pt:
    print("\n✅ Perfect alignment!")

# Side-by-side sample
print("\n--- Fragment 1 comparison ---")
print(f"PT paragraphs: {len(pt['1']['paragraphs'])}")
print(f"EN paragraphs: {len(english_sections.get('1', []))}")

Portuguese fragments: 481
English fragments: 481

✅ Perfect alignment!

--- Fragment 1 comparison ---
PT paragraphs: 6
EN paragraphs: 9
